<a href="https://colab.research.google.com/github/Diggi14/project_Property2/blob/main/recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [292]:
!git clone https://github.com/Diggi14/project_Property2.git

fatal: destination path 'project_Property2' already exists and is not an empty directory.


In [293]:
import pandas as pd
import numpy as np

In [294]:
df=pd.read_csv('/content/project_Property2/recommender.csv')
df.head(4)

,PREFERENCE,PROPERTY_TYPE,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,SOCIETY_NAME,LOCALITY_WO_CITY,FEATURES_LIST,stories,nearby_facilities
0,S,Residential Apartment,4.0,4.0,4.0,8766.0,Semi-furnished,5-10,14.0,3434.0,2.63,Alpha Corp GurgaonOne 84,Sector 84,"['Intercom Facility', 'Lift(s)', 'Water purifi...",single,"['Bank/ATM', 'Park', 'School/College', 'Hospit..."
1,S,Residential Apartment,4.0,4.0,3.0,21176.0,Semi-furnished,1-5,7.0,2870.0,3.60,DLF The Ultima,Sector 81,"['Security / Fire Alarm', 'Feng Shui / Vaastu ...",single,"['Railway', 'School/College', 'Metro', 'Hospit..."
2,S,Residential Apartment,3.0,3.0,3.0,13740.0,Semi-furnished,1-5,14.0,2802.0,3.85,Experion Windchants,Sector 112,"['Power Back-up', 'Feng Shui / Vaastu Complian...",single,"['Railway', 'School/College', 'Hospital', 'Air..."
3,S,Residential Apartment,3.0,4.0,4.0,8515.0,Semi-furnished,1-5,4.0,2290.0,1.95,ATS Triumph,Sector 104,"['Security / Fire Alarm', 'Power Back-up', 'In...",single,"['Park', 'Railway', 'School/College', 'Metro',..."


In [295]:
df=df[df['PROPERTY_TYPE']=='Residential Apartment']
df = df.reset_index(drop=True)

In [296]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [297]:
import re

df['nearby_facilities'] = df['nearby_facilities'].apply(lambda x: re.findall(r"'(.*?)'", x))

In [298]:
df['facility_token']=df['nearby_facilities'].apply(lambda x:' '.join(x))

In [299]:
df['FEATURES_LIST'] = df['FEATURES_LIST'].apply(lambda x: re.findall(r"'(.*?)'", x))

In [300]:
df['feature_token']=df['FEATURES_LIST'].apply(lambda x:' '.join(x))

In [301]:
df.drop(['nearby_facilities','FEATURES_LIST'],inplace=True,axis=1)

In [302]:
df.head(4)

,PREFERENCE,PROPERTY_TYPE,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,SOCIETY_NAME,LOCALITY_WO_CITY,stories,facility_token,feature_token
0,S,Residential Apartment,4.0,4.0,4.0,8766.0,Semi-furnished,5-10,14.0,3434.0,2.63,Alpha Corp GurgaonOne 84,Sector 84,single,Bank/ATM Park School/College Hospital,Intercom Facility Lift(s) Water purifier Maint...
1,S,Residential Apartment,4.0,4.0,3.0,21176.0,Semi-furnished,1-5,7.0,2870.0,3.60,DLF The Ultima,Sector 81,single,Railway School/College Metro Hospital Airport,Security / Fire Alarm Feng Shui / Vaastu Compl...
2,S,Residential Apartment,3.0,3.0,3.0,13740.0,Semi-furnished,1-5,14.0,2802.0,3.85,Experion Windchants,Sector 112,single,Railway School/College Hospital Airport,Power Back-up Feng Shui / Vaastu Compliant Int...
3,S,Residential Apartment,3.0,4.0,4.0,8515.0,Semi-furnished,1-5,4.0,2290.0,1.95,ATS Triumph,Sector 104,single,Park Railway School/College Metro Hospital Air...,Security / Fire Alarm Power Back-up Intercom F...


In [303]:
from sklearn.feature_extraction.text import CountVectorizer

In [304]:
vectorizer = CountVectorizer(stop_words='english')
matrix1=vectorizer.fit_transform(df['facility_token'])
matrix2=vectorizer.fit_transform(df['feature_token'])

In [305]:
matrix1.toarray().shape

(6302, 9)

In [306]:
from sklearn.metrics.pairwise import cosine_similarity

In [307]:
cosine_sim=cosine_similarity(matrix1,matrix1)

In [308]:
cosine_sim

array([[1.        , 0.5       , 0.54772256, ..., 0.40824829, 0.40824829,
        0.40824829],
       [0.5       , 1.        , 0.91287093, ..., 0.40824829, 0.40824829,
        0.40824829],
       [0.54772256, 0.91287093, 1.        , ..., 0.4472136 , 0.4472136 ,
        0.4472136 ],
       ...,
       [0.40824829, 0.40824829, 0.4472136 , ..., 1.        , 1.        ,
        1.        ],
       [0.40824829, 0.40824829, 0.4472136 , ..., 1.        , 1.        ,
        1.        ],
       [0.40824829, 0.40824829, 0.4472136 , ..., 1.        , 1.        ,
        1.        ]])

# ***Facility***

In [309]:
def recomend_properties(property_name,cosine_sim=cosine_sim):
  idx=df[df['SOCIETY_NAME']==property_name].index.tolist()[0]
  sim_score=list(enumerate(cosine_sim[idx]))
  sim_score=sorted(sim_score,key=lambda x:x[1],reverse=True)
  index=[]
  prop=set()
  for i,j in sim_score:
    society_name = df.loc[i, 'SOCIETY_NAME']
    if society_name not in prop and society_name !=property_name:
            index.append(i)
            prop.add(society_name)
    if len(index) >= 10:
      break
  return df.loc[index]

In [310]:
recomend_properties('DLF The Ultima',cosine_sim)

,PREFERENCE,PROPERTY_TYPE,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,SOCIETY_NAME,LOCALITY_WO_CITY,stories,facility_token,feature_token
14,S,Residential Apartment,3.0,3.0,3.0,12612.000000,Semi-furnished,Under Construction,12.0,1665.0,2.10,M3M Capital,Sector 113 Gurgaon,single,Railway School/College Metro Hospital Airport,Lift(s) Swimming Pool Fitness Centre / GYM Clu...
18,S,Residential Apartment,3.0,3.0,2.0,18138.000000,Semi-furnished,1-5,9.0,3363.0,6.10,M3M Golfestate,Sector 65,single,Railway School/College Metro Hospital Airport,Centrally Air Conditioned Water purifier Secur...
35,S,Residential Apartment,4.0,4.0,3.0,10109.000000,Semi-furnished,Under Construction,10.0,2913.0,2.94,Sobha City,Sector 108,single,Railway School/College Metro Hospital Airport,Power Back-up Intercom Facility Lift(s) Swimmi...
38,S,Residential Apartment,3.0,3.0,3.0,7282.000000,Semi-furnished,Under Construction,7.0,1900.0,1.38,Godrej Meridien,Sector 106,single,Railway School/College Metro Hospital Airport,Intercom Facility Lift(s) Maintenance Staff Sw...
46,S,Residential Apartment,2.0,2.0,3.0,12078.000000,Semi-furnished,0-1,5.0,1573.0,1.90,La Vida by Tata Housing,Sector 113 Gurgaon,single,Railway School/College Metro Hospital Airport,Feng Shui / Vaastu Compliant Security / Fire A...
60,S,Residential Apartment,3.0,2.0,2.0,10669.000000,Semi-furnished,Under Construction,28.0,1359.0,1.45,Hero Homes,Sector 104,single,Railway School/College Metro Hospital Airport,Centrally Air Conditioned Water purifier Secur...
72,S,Residential Apartment,3.0,4.0,4.0,9304.000000,Semi-furnished,0-1,8.0,1999.0,1.86,Pareena Mi Casa,Sector 68,single,Railway School/College Metro Hospital Airport,Water purifier Centrally Air Conditioned Secur...
107,S,Residential Apartment,3.0,3.0,3.0,26595.744681,Semi-furnished,1-5,8.0,2350.0,6.25,Central Park Resorts,Sector 48,single,Railway School/College Metro Hospital Airport,Power Back-up Intercom Facility Lift(s) Piped-...
192,S,Residential Apartment,3.0,3.0,2.0,7204.000000,Unfurnished,1-5,14.0,1735.0,1.25,Vatika Gurgaon 21,Sector 83,single,Railway School/College Metro Hospital Airport,Feng Shui / Vaastu Compliant Lift(s) Maintenan...
284,S,Residential Apartment,3.0,3.0,4.0,21788.000000,Semi-furnished,Under Construction,3.0,1331.0,2.90,Birla Navya,Sector 63A,single,Railway School/College Metro Hospital Airport,Centrally Air Conditioned Security / Fire Alar...


# ***location***

In [311]:
cosine_sim2=cosine_similarity(matrix2,matrix2)

In [312]:
def recomend_properties2(property_name,cosine_sim=cosine_sim2):
  idx=df[df['SOCIETY_NAME']=='DLF The Ultima'].index.tolist()[0]
  sim_score=list(enumerate(cosine_sim[idx]))
  sim_score=sorted(sim_score,key=lambda x:x[1],reverse=True)
  index=[]
  prop=set()
  for i,j in sim_score:
    society_name = df.loc[i, 'SOCIETY_NAME']
    if society_name not in prop and society_name !=property_name:
            index.append(i)
            prop.add(society_name)
    if len(index) >= 10:
      break
  return df.loc[index]

In [313]:
recomend_properties2('DLF The Ultima',cosine_sim2)

,PREFERENCE,PROPERTY_TYPE,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,SOCIETY_NAME,LOCALITY_WO_CITY,stories,facility_token,feature_token
41,S,Residential Apartment,2.0,3.0,3.0,14965.0,Semi-furnished,1-5,10.0,1470.0,2.20,Ireo Skyon,Sector 60,single,Park School/College Metro Hospital Airport,Centrally Air Conditioned Water purifier Secur...
474,S,Residential Apartment,3.0,3.0,4.0,12045.0,Semi-furnished,0-1,9.0,2092.0,2.52,DLF The Primus,Sector 82A,single,Railway School/College Hospital Airport,Security / Fire Alarm Feng Shui / Vaastu Compl...
1381,S,Residential Apartment,3.0,3.0,2.0,8400.0,Furnished,1-5,16.0,1726.0,1.45,DLF Regal Gardens,Sector 90,single,Bank/ATM School/College Hospital,Security / Fire Alarm Feng Shui / Vaastu Compl...
1581,S,Residential Apartment,3.0,4.0,3.0,14541.0,Semi-furnished,1-5,15.0,2400.0,3.49,M3M Merlin,Sector 67,single,Park School/College Hospital Airport,Centrally Air Conditioned Water purifier Secur...
1907,S,Residential Apartment,2.0,2.0,2.0,12000.0,Semi-furnished,Under Construction,9.0,1478.0,1.77,Godrej Arista,Sector 79,single,,Security / Fire Alarm Feng Shui / Vaastu Compl...
2358,S,Residential Apartment,4.0,4.0,4.0,20683.0,Semi-furnished,1-5,15.0,3626.0,7.50,Mahindra Luminare,Sector 59,single,Park Bank/ATM School/College Metro Hospital Ai...,Centrally Air Conditioned Water purifier Secur...
2790,S,Residential Apartment,3.0,3.0,3.0,13631.0,Semi-furnished,Under Construction,9.0,2054.0,2.80,M3M Skycity,Sector 65,single,Park School/College Metro Hospital Airport,Lift(s) Swimming Pool Park Fitness Centre / GY...
3667,S,Residential Apartment,2.0,2.0,2.0,9500.0,Unfurnished,1-5,7.0,1260.0,0.76,M3M The Marina,Sector 68,single,Park School/College Hospital Airport,Security / Fire Alarm Feng Shui / Vaastu Compl...
3689,S,Residential Apartment,3.0,3.0,3.0,10167.0,Semi-furnished,0-1,9.0,1967.0,2.00,Ireo The Corridors,Sector 67A,single,Metro School/College Airport Park,Security / Fire Alarm Power Back-up Feng Shui ...
5592,S,Residential Apartment,3.0,3.0,4.0,6293.0,Semi-furnished,1-5,18.0,1430.0,0.90,Mapsko Casa Bella,Sector 82,single,Park School/College Hospital Airport,Centrally Air Conditioned Water purifier Secur...


In [314]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6302 entries, 0 to 6305
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PREFERENCE           6302 non-null   object 
 1   PROPERTY_TYPE        6302 non-null   object 
 2   BEDROOM_NUM          6302 non-null   float64
 3   BATHROOM_NUM         6302 non-null   float64
 4   BALCONY_NUM          6302 non-null   float64
 5   PRICE_PER_UNIT_AREA  6302 non-null   float64
 6   FURNISH              6302 non-null   object 
 7   AGE                  6302 non-null   object 
 8   FLOOR_NUM            6302 non-null   float64
 9   SUPERBUILTUP_SQFT    6302 non-null   float64
 10  PRICE                6302 non-null   float64
 11  SOCIETY_NAME         6302 non-null   object 
 12  LOCALITY_WO_CITY     6302 non-null   object 
 13  stories              6302 non-null   object 
 14  facility_token       6302 non-null   object 
 15  feature_token        6302 non-null   object

In [315]:
from sklearn.preprocessing import StandardScaler


# **price and area**

In [316]:
features = df[['BEDROOM_NUM','PRICE_PER_UNIT_AREA', 'PRICE']].copy()
sc=StandardScaler()
scaled_features = sc.fit_transform(features)
cosine_sim3 = cosine_similarity(scaled_features, scaled_features)

In [317]:
def recomend_properties3(property_name,cosine_sim=cosine_sim):
  idx=df[df['SOCIETY_NAME']==property_name].index.tolist()[0]
  sim_score=list(enumerate(cosine_sim[idx]))
  sim_score=sorted(sim_score,key=lambda x:x[1],reverse=True)
  index=[]
  prop=set()
  for i,j in sim_score:
    society_name = df.loc[i, 'SOCIETY_NAME']
    if society_name not in prop and society_name !=property_name:
            index.append(i)
            prop.add(society_name)
    if len(index) >= 10:
      break
  return df.loc[index]

In [318]:
recomend_properties3('DLF The Ultima',cosine_sim3)

,PREFERENCE,PROPERTY_TYPE,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,SOCIETY_NAME,LOCALITY_WO_CITY,stories,facility_token,feature_token
1151,S,Residential Apartment,4.0,4.0,4.0,21176.0,Semi-furnished,1-5,0.0,1700.0,3.600,Emaar Emerald Hills,Sector 65,single,Park Railway School/College Metro Hospital Air...,Security / Fire Alarm Feng Shui / Vaastu Compl...
1547,S,Residential Apartment,3.0,3.0,3.0,12127.0,Semi-furnished,1-5,4.0,2350.0,2.850,Central Park 2 Bellevue,Sector 48,single,Bank/ATM Hospital,Lift(s) Swimming Pool Park Shopping Centre Fit...
5179,S,Residential Apartment,4.0,5.0,4.0,14315.0,Semi-furnished,5-10,1.0,2410.0,3.450,Emaar MGF Palm Terraces,Sector 66,single,Metro School/College Hospital Airport,Power Back-up Feng Shui / Vaastu Compliant Int...
5716,S,Residential Apartment,3.0,2.0,4.0,12813.0,Semi-furnished,5-10,3.0,1717.0,2.200,Raisina Residency,Sector 59,single,Bank/ATM Metro School/College Park,Security / Fire Alarm Feng Shui / Vaastu Compl...
663,S,Residential Apartment,4.0,4.0,4.0,19200.0,Semi-furnished,1-5,12.0,2100.0,3.360,Emaar MGF The Palm Drive,Sector 66,single,Park Railway School/College Metro Hospital Air...,Feng Shui / Vaastu Compliant Security / Fire A...
953,S,Residential Apartment,4.0,5.0,4.0,22486.0,Semi-furnished,1-5,10.0,2905.0,4.250,Tata Primanti,Sector 72,single,Metro School/College Hospital,Security / Fire Alarm Feng Shui / Vaastu Compl...
5715,S,Residential Apartment,3.0,3.0,4.0,11320.0,Semi-furnished,1-5,8.0,2385.0,2.700,Ireo Skyon,Sector 60,single,Park School/College Metro Hospital Airport,Centrally Air Conditioned Water purifier Secur...
2397,S,Residential Apartment,3.0,4.0,4.0,13319.0,Semi-furnished,Under Construction,7.0,1877.0,2.500,BPTP Terra,Sector 37D,single,Park School/College Hospital Airport,Security / Fire Alarm Lift(s) Maintenance Staf...
5030,S,Residential Apartment,2.0,2.0,2.0,6443.0,Unfurnished,1-5,5.0,950.0,0.385,Breez Global Heights,Sohna,single,Park School/College Hospital Airport,Feng Shui / Vaastu Compliant Security / Fire A...
301,S,Residential Apartment,4.0,4.0,4.0,19062.0,Semi-furnished,1-5,2.0,2606.0,3.050,DLF The Primus,Sector 82A,single,Railway School/College Hospital Airport,Centrally Air Conditioned Water purifier Secur...


# ***final***

In [319]:
def final_recomendation(property_name,cosine_sim,cosine_sim2,cosine_sim3,weights):
  weights = np.array(weights)
  weights = weights / weights.sum()
  f_cosine=weights[0]*cosine_sim+weights[1]*cosine_sim2+weights[2]*cosine_sim3
  idx=df[df['SOCIETY_NAME']==property_name].index.tolist()[0]
  sim_score=list(enumerate(cosine_sim[idx]))
  sim_score=sorted(sim_score,key=lambda x:x[1],reverse=True)
  index=[]
  prop=set()
  for i,j in sim_score:
    society_name = df.loc[i, 'SOCIETY_NAME']
    if society_name not in prop and society_name !=property_name:
            index.append(i)
            prop.add(society_name)
    if len(index) >= 10:
      break
  return df.loc[index]


In [320]:
df[df['SOCIETY_NAME']=='DLF The Ultima'].head(1)

,PREFERENCE,PROPERTY_TYPE,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,SOCIETY_NAME,LOCALITY_WO_CITY,stories,facility_token,feature_token
1,S,Residential Apartment,4.0,4.0,3.0,21176.0,Semi-furnished,1-5,7.0,2870.0,3.6,DLF The Ultima,Sector 81,single,Railway School/College Metro Hospital Airport,Security / Fire Alarm Feng Shui / Vaastu Compl...


In [321]:
final_recomendation('DLF The Ultima',cosine_sim,cosine_sim2,cosine_sim3,[0,0,5])

,PREFERENCE,PROPERTY_TYPE,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,SOCIETY_NAME,LOCALITY_WO_CITY,stories,facility_token,feature_token
14,S,Residential Apartment,3.0,3.0,3.0,12612.000000,Semi-furnished,Under Construction,12.0,1665.0,2.10,M3M Capital,Sector 113 Gurgaon,single,Railway School/College Metro Hospital Airport,Lift(s) Swimming Pool Fitness Centre / GYM Clu...
18,S,Residential Apartment,3.0,3.0,2.0,18138.000000,Semi-furnished,1-5,9.0,3363.0,6.10,M3M Golfestate,Sector 65,single,Railway School/College Metro Hospital Airport,Centrally Air Conditioned Water purifier Secur...
35,S,Residential Apartment,4.0,4.0,3.0,10109.000000,Semi-furnished,Under Construction,10.0,2913.0,2.94,Sobha City,Sector 108,single,Railway School/College Metro Hospital Airport,Power Back-up Intercom Facility Lift(s) Swimmi...
38,S,Residential Apartment,3.0,3.0,3.0,7282.000000,Semi-furnished,Under Construction,7.0,1900.0,1.38,Godrej Meridien,Sector 106,single,Railway School/College Metro Hospital Airport,Intercom Facility Lift(s) Maintenance Staff Sw...
46,S,Residential Apartment,2.0,2.0,3.0,12078.000000,Semi-furnished,0-1,5.0,1573.0,1.90,La Vida by Tata Housing,Sector 113 Gurgaon,single,Railway School/College Metro Hospital Airport,Feng Shui / Vaastu Compliant Security / Fire A...
60,S,Residential Apartment,3.0,2.0,2.0,10669.000000,Semi-furnished,Under Construction,28.0,1359.0,1.45,Hero Homes,Sector 104,single,Railway School/College Metro Hospital Airport,Centrally Air Conditioned Water purifier Secur...
72,S,Residential Apartment,3.0,4.0,4.0,9304.000000,Semi-furnished,0-1,8.0,1999.0,1.86,Pareena Mi Casa,Sector 68,single,Railway School/College Metro Hospital Airport,Water purifier Centrally Air Conditioned Secur...
107,S,Residential Apartment,3.0,3.0,3.0,26595.744681,Semi-furnished,1-5,8.0,2350.0,6.25,Central Park Resorts,Sector 48,single,Railway School/College Metro Hospital Airport,Power Back-up Intercom Facility Lift(s) Piped-...
192,S,Residential Apartment,3.0,3.0,2.0,7204.000000,Unfurnished,1-5,14.0,1735.0,1.25,Vatika Gurgaon 21,Sector 83,single,Railway School/College Metro Hospital Airport,Feng Shui / Vaastu Compliant Lift(s) Maintenan...
284,S,Residential Apartment,3.0,3.0,4.0,21788.000000,Semi-furnished,Under Construction,3.0,1331.0,2.90,Birla Navya,Sector 63A,single,Railway School/College Metro Hospital Airport,Centrally Air Conditioned Security / Fire Alar...
